Fix scikit-learn version incompatibility with snowflake-ml SimpleImputer
*Co-authored with CoCo*

In [ ]:
%%sql -r dataframe_3
USE DATABASE EUROPEAN_SOCCER_DB;
USE SCHEMA MARTS_ML;
USE WAREHOUSE ML_WH;

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE TABLE ML_PLAYER_RATING_DATASET AS

SELECT

    PLAYER_API_ID,
    LAST_ATB_UPDATE_DATE,

    CROSSING,
    FINISHING,
    HEADING_ACCURACY,
    SHORT_PASSING,
    VOLLEYS,
    DRIBBLING,
    CURVE,
    FREE_KICK_ACCURACY,
    LONG_PASSING,
    BALL_CONTROL,

    ACCELERATION,
    SPRINT_SPEED,
    AGILITY,
    REACTIONS,
    BALANCE,
    SHOT_POWER,
    JUMPING,
    STAMINA,
    STRENGTH,
    LONG_SHOTS,

    AGGRESSION,
    INTERCEPTIONS,
    POSITIONING,
    VISION,
    PENALTIES,

    MARKING,
    STANDING_TACKLE,
    SLIDING_TACKLE,

    GK_DIVING,
    GK_HANDLING,
    GK_KICKING,
    GK_POSITIONING,
    GK_REFLEXES,

    OVERALL_RATING

FROM EUROPEAN_SOCCER_DB.MARTS_CORE.FACT_PLAYERS_ATTRIBUTES

WHERE OVERALL_RATING IS NOT NULL;

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

print(session.get_current_database())
print(session.get_current_schema())
print(session.get_current_warehouse())

In [ ]:
session.add_packages(
    "pandas",
    "scikit-learn",
    "snowflake-snowpark-python"
)

print(session.get_packages())

In [ ]:
table_name = "EUROPEAN_SOCCER_DB.MARTS_ML.ML_PLAYER_RATING_DATASET"

df = session.table(table_name)

print(df.count())

In [ ]:
target_column = "OVERALL_RATING"

id_columns = [
    "PLAYER_API_ID",
    "LAST_ATB_UPDATE_DATE"
]

feature_columns = [
    "CROSSING",
    "FINISHING",
    "HEADING_ACCURACY",
    "SHORT_PASSING",
    "VOLLEYS",
    "DRIBBLING",
    "CURVE",
    "FREE_KICK_ACCURACY",
    "LONG_PASSING",
    "BALL_CONTROL",
    "ACCELERATION",
    "SPRINT_SPEED",
    "AGILITY",
    "REACTIONS",
    "BALANCE",
    "SHOT_POWER",
    "JUMPING",
    "STAMINA",
    "STRENGTH",
    "LONG_SHOTS",
    "AGGRESSION",
    "INTERCEPTIONS",
    "POSITIONING",
    "VISION",
    "PENALTIES",
    "MARKING",
    "STANDING_TACKLE",
    "SLIDING_TACKLE",
    "GK_DIVING",
    "GK_HANDLING",
    "GK_KICKING",
    "GK_POSITIONING",
    "GK_REFLEXES"
]

output_column = "PREDICTED_OVERALL_RATING"

In [ ]:
from snowflake.snowpark.functions import col

df_ml = df.select(
    col("PLAYER_API_ID"),
    col("LAST_ATB_UPDATE_DATE"),
    *[
        col(column).cast("double").alias(column)
        for column in feature_columns
    ],
    col(target_column).cast("double").alias(target_column)
)

df_ml.print_schema()
df_ml.show(5)

In [ ]:
train_raw_df, test_raw_df = df_ml.random_split(
    weights=[0.80, 0.20],
    seed=42
)

print("Linhas de treino:", train_raw_df.count())
print("Linhas de teste:", test_raw_df.count())

In [ ]:
!pip install "numpy<2.5" "scikit-learn<1.8" --quiet

from snowflake.ml.modeling.impute import SimpleImputer

imputed_feature_columns = [
    f"{column}_IMPUTED"
    for column in feature_columns
]

imputer = SimpleImputer(
    input_cols=feature_columns,
    output_cols=imputed_feature_columns,
    strategy="median"
)

imputer.fit(train_raw_df)

train_df = imputer.transform(train_raw_df)
test_df = imputer.transform(test_raw_df)

In [ ]:
from snowflake.snowpark.functions import col

df_ml = df.select(
    col("PLAYER_API_ID"),
    col("LAST_ATB_UPDATE_DATE"),
    *[
        col(column).cast("double").alias(column)
        for column in feature_columns
    ],
    col(target_column).cast("double").alias(target_column)
)

df_ml.print_schema()
df_ml.show(5)

In [ ]:
from snowflake.ml.modeling.ensemble import RandomForestRegressor

model = RandomForestRegressor(

    input_cols=imputed_feature_columns,

    label_cols=[target_column],

    output_cols=[output_column],

    n_estimators=100,

    max_depth=12,

    min_samples_leaf=2,

    random_state=42

)

In [ ]:
train_pdf = train_df.to_pandas()
train_pdf.columns = train_pdf.columns.astype(object)
model.fit(train_pdf)

In [ ]:
test_pdf = test_df.to_pandas()
test_pdf.columns = test_pdf.columns.astype(object)
predictions_df = model.predict(test_pdf)

In [ ]:
predictions_df[["PLAYER_API_ID", "LAST_ATB_UPDATE_DATE", target_column, output_column]].head(10)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_true = predictions_df[target_column]
y_pred = predictions_df[output_column]

mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)
rmse = mse ** 0.5

print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")

In [ ]:
import pandas as pd
from datetime import datetime
from snowflake.connector.pandas_tools import write_pandas

result_df = predictions_df[["PLAYER_API_ID", "LAST_ATB_UPDATE_DATE", target_column, output_column]].copy()
result_df["ABSOLUTE_ERROR"] = (result_df[target_column] - result_df[output_column]).abs()
result_df["PREDICTED_AT"] = datetime.now()

connection = session.connection

session.sql("CREATE OR REPLACE TABLE EUROPEAN_SOCCER_DB.MARTS_ML.FACT_PLAYER_RATING_PREDICTIONS (PLAYER_API_ID NUMBER, LAST_ATB_UPDATE_DATE DATE, OVERALL_RATING DOUBLE, PREDICTED_OVERALL_RATING DOUBLE, ABSOLUTE_ERROR DOUBLE, PREDICTED_AT TIMESTAMP)").collect()

write_pandas(
    conn=connection,
    df=result_df,
    table_name="FACT_PLAYER_RATING_PREDICTIONS",
    database="EUROPEAN_SOCCER_DB",
    schema="MARTS_ML",
    quote_identifiers=False
)

print(f"Wrote {len(result_df)} rows to FACT_PLAYER_RATING_PREDICTIONS")